In [ ]:
import pkg_resources
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.core.exceptions import HttpResponseError
import mltable
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score , f1_score , confusion_matrix
from azure.ai.ml.entities import Data
from azure.ai.ml import automl
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.automl import ClassificationPrimaryMetrics
from azure.ai.ml import Input
from ucimlrepo import fetch_ucirepo
from mltable import DataType
import mlflow
import subprocess
import warnings 
warnings.filterwarnings('ignore')

C:\Users\leste\AppData\Local\Temp\ipykernel_33472\2023927064.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
c:\Users\leste\AppData\Local\PythonEnvs\MLOps-Python-3.12.14\Lib\site-packages\mlflow\__init__.py:41: UserWarning: Versions of mlflow (3.18.0) and child packages mlflow-skinny (3.15.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()
c:\Users\leste\AppData\Local\PythonEnvs\MLOps-Python-3.12.14\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
subprocess.run('azd auth login')

CompletedProcess(args='azd auth login', returncode=0)

> Authenticating with Azure Machine Learning Studio

In [3]:
ml_client = MLClient(
    credential = DefaultAzureCredential(),
    subscription_id = "48f83b14-32a8-4911-b629-ea5aead7fb29",
    resource_group_name =  "lm1610-rg",
    workspace_name =  "lm_portfolio"
)

In [4]:
banking_data = fetch_ucirepo(id = 222)

In [5]:
X = banking_data.data.features
Y = banking_data.data.targets

In [6]:
banking_df = pd.concat([X,Y], axis = 1)

In [7]:
banking_df

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,no,825,no,no,cellular,17,nov,977,3,-1,0,NaN,yes
45207,71,retired,divorced,primary,no,1729,no,no,cellular,17,nov,456,2,-1,0,NaN,yes
45208,72,retired,married,secondary,no,5715,no,no,cellular,17,nov,1127,5,184,3,success,yes
45209,57,blue-collar,married,secondary,no,668,no,no,telephone,17,nov,508,4,-1,0,NaN,no


In [8]:
banking_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   age          45211 non-null  int64
 1   job          44923 non-null  str  
 2   marital      45211 non-null  str  
 3   education    43354 non-null  str  
 4   default      45211 non-null  str  
 5   balance      45211 non-null  int64
 6   housing      45211 non-null  str  
 7   loan         45211 non-null  str  
 8   contact      32191 non-null  str  
 9   day_of_week  45211 non-null  int64
 10  month        45211 non-null  str  
 11  duration     45211 non-null  int64
 12  campaign     45211 non-null  int64
 13  pdays        45211 non-null  int64
 14  previous     45211 non-null  int64
 15  poutcome     8252 non-null   str  
 16  y            45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 7.7 MB


In [9]:
banking_cleaned_df = banking_df.fillna(
    {
        'poutcome' : 'unknown',
        'contact' : 'unknown',
        'job' : 'unknown',
        'education' : 'cannot be determined'
    }
).rename(columns = {
    'y' : 'Subscribed'
})

In [10]:
banking_cleaned_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   age          45211 non-null  int64
 1   job          45211 non-null  str  
 2   marital      45211 non-null  str  
 3   education    45211 non-null  str  
 4   default      45211 non-null  str  
 5   balance      45211 non-null  int64
 6   housing      45211 non-null  str  
 7   loan         45211 non-null  str  
 8   contact      45211 non-null  str  
 9   day_of_week  45211 non-null  int64
 10  month        45211 non-null  str  
 11  duration     45211 non-null  int64
 12  campaign     45211 non-null  int64
 13  pdays        45211 non-null  int64
 14  previous     45211 non-null  int64
 15  poutcome     45211 non-null  str  
 16  Subscribed   45211 non-null  str  
dtypes: int64(7), str(10)
memory usage: 8.1 MB


In [26]:
X_features = banking_cleaned_df.drop(['Subscribed'], axis = 1)
Y_features = banking_cleaned_df['Subscribed']

In [27]:
X_features_updated = pd.get_dummies(X_features).replace({True : 1, False : 0})

In [28]:
Y_features_updated = Y_features.astype('object') 

In [29]:
model = LogisticRegression()

In [30]:
Y_features.shape

(45211,)

In [31]:
X_train , X_temp , Y_train , Y_temp = train_test_split(X_features_updated,Y_features_updated , test_size = 0.5 , random_state = 45)
X_test ,  X_val , Y_test , Y_val = train_test_split(X_temp, Y_temp , test_size = 0.4 , random_state = 45)

In [33]:
type_of_target(Y_train)

'binary'

In [ ]:
accuracy_score(Y_train,y_train_pred)

0.8917938509179385

In [54]:
accuracy_score(Y_val,y_val_pred)

0.8932876257879022

In [52]:
confusion_matrix(Y_train,y_train_pred)

array([[19547,   411],
       [ 2035,   612]])

### Integrating MLflow

In [63]:
mlflow.set_tracking_uri('azureml://centralindia.api.azureml.ms/mlflow/v1.0/subscriptions/48f83b14-32a8-4911-b629-ea5aead7fb29/resourceGroups/lm1610-rg/providers/Microsoft.MachineLearningServices/workspaces/lm_portfolio')

In [64]:
mlflow.set_experiment(experiment_name = 'Python code Classification job')

2026/11/16 16:22:06 INFO mlflow.tracking.fluent: Experiment with name 'Python code Classification job' does not exist. Creating a new experiment.


<Experiment: artifact_location='', creation_time=1794831730921, effective_trace_archival_retention=None, experiment_id='7e3c8aaf-801d-4b9f-a3cf-7cf1ed701657', last_update_time=None, lifecycle_stage='active', name='Python code Classification job', tags={}, trace_location=None, workspace='default'>

In [69]:
with mlflow.start_run():
    model = LogisticRegression()
    model.fit(X_train , Y_train)
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    training_accuracy = accuracy_score(Y_train,y_train_pred)
    validation_accuracy = accuracy_score(Y_val,y_val_pred)

    mlflow.log_params(
        {
            'Model Used' : model,
            'Training Accuracy' : training_accuracy,
            'Validation Accuracy' : validation_accuracy
        }
    )

    
    

🏃 View run magenta_lime_4pfxn79n at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/48f83b14-32a8-4911-b629-ea5aead7fb29/resourceGroups/lm1610-rg/providers/Microsoft.MachineLearningServices/workspaces/lm_portfolio/#/experiments/7e3c8aaf-801d-4b9f-a3cf-7cf1ed701657/runs/6cd44160-21b3-4e25-b21f-d31cdfcb523b
🧪 View experiment at: https://centralindia.api.azureml.ms/mlflow/v2.0/subscriptions/48f83b14-32a8-4911-b629-ea5aead7fb29/resourceGroups/lm1610-rg/providers/Microsoft.MachineLearningServices/workspaces/lm_portfolio/#/experiments/7e3c8aaf-801d-4b9f-a3cf-7cf1ed701657


In [ ]:
mlflow.log_params(
    {
        'dataset' : banking_cleaned_df,
        'model' : 'LogisticRegression',
        'Accuracy': accuracy_score, 
        
    }
)